# 🎯 YOLO26 얼굴 감지 파인튜닝 v2

## v1 → v2 변경사항
- ✅ `train_xxx.json` + `val_xxx.json` **둘 다 활용**
- ✅ JSON에 명시된 split 그대로 유지 (임의 분할 X)
- ✅ 더 정확한 평가 가능

## 작업 흐름

```
HSEmotion JSON (10개 파일)
├── train_anger.json    ┐
├── train_blank.json    │
├── train_happy.json    │  →  YOLO train 데이터셋
├── train_panic.json    │
└── train_sadness.json  ┘

├── val_anger.json      ┐
├── val_blank.json      │
├── val_happy.json      │  →  YOLO val 데이터셋 (평가용)
├── val_panic.json      │
└── val_sadness.json    ┘
                                  ↓
                          YOLO26 파인튜닝
                                  ↓
                          best_face_model.pt
```

## 1️⃣ 환경 설정

In [ ]:
# 패키지 설치
!pip install ultralytics tqdm pillow -q

print("✅ 패키지 설치 완료")

In [ ]:
import os

# ═══════════════════════════════════════════════════
# 📁 경로 설정
# ═══════════════════════════════════════════════════

# 입력 데이터
JSON_DIR        = "/workspace/user5/relabeled"            # JSON 파일 10개 위치
IMAGE_BASE_DIR  = "/workspace/user5/img/img(classify)"    # 원본 이미지 폴더

# 출력 (YOLO 형식 데이터셋)
YOLO_DATASET_DIR = "/workspace/user5/yolo_face_dataset"

# 학습 결과 저장
MODEL_OUTPUT_DIR = "/workspace/user5/yolo_finetuned"

# 학습 설정
EPOCHS     = 50
BATCH_SIZE = 32
IMG_SIZE   = 640
DEVICE     = 0

print(f"📁 JSON: {JSON_DIR}")
print(f"📁 이미지: {IMAGE_BASE_DIR}")
print(f"📁 YOLO 데이터셋: {YOLO_DATASET_DIR}")
print(f"📁 모델 저장: {MODEL_OUTPUT_DIR}")

## 2️⃣ HSEmotion JSON → YOLO 형식 변환

10개 JSON 파일 (train 5개 + val 5개)을 모두 처리해서 YOLO 데이터셋 생성

In [ ]:
import json
import shutil
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

# 처리할 JSON 파일 매핑 (split, emotion) → 파일명
JSON_FILES = {
    # Train
    ("train", "anger"):   "train_anger.json",
    ("train", "blank"):   "train_blank.json",
    ("train", "happy"):   "train_happy.json",
    ("train", "panic"):   "train_panic.json",
    ("train", "sadness"): "train_sadness.json",
    # Val
    ("val", "anger"):     "val_anger.json",
    ("val", "blank"):     "val_blank.json",
    ("val", "happy"):     "val_happy.json",
    ("val", "panic"):     "val_panic.json",
    ("val", "sadness"):   "val_sadness.json",
}


def convert_bbox_to_yolo(bbox, img_w, img_h):
    """HSEmotion bbox → YOLO 형식 (cx, cy, w, h, 0~1 정규화)"""
    min_x, min_y = bbox["minX"], bbox["minY"]
    max_x, max_y = bbox["maxX"], bbox["maxY"]

    # 이미지 범위 보정
    min_x = max(0, min(min_x, img_w))
    min_y = max(0, min(min_y, img_h))
    max_x = max(0, min(max_x, img_w))
    max_y = max(0, min(max_y, img_h))

    cx = (min_x + max_x) / 2 / img_w
    cy = (min_y + max_y) / 2 / img_h
    w  = (max_x - min_x) / img_w
    h  = (max_y - min_y) / img_h

    return cx, cy, w, h


def find_image(filename, split, emotion):
    """이미지 파일 경로 탐색"""
    candidates = [
        os.path.join(IMAGE_BASE_DIR, split, emotion, filename),
        os.path.join(IMAGE_BASE_DIR, "train", emotion, filename),
        os.path.join(IMAGE_BASE_DIR, "val",   emotion, filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def setup_yolo_folders():
    """YOLO 데이터셋 폴더 구조 생성"""
    for split in ["train", "val"]:
        os.makedirs(os.path.join(YOLO_DATASET_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(YOLO_DATASET_DIR, "labels", split), exist_ok=True)
    print(f"✅ YOLO 폴더 구조 생성: {YOLO_DATASET_DIR}")


print("✅ 변환 함수 준비 완료")

In [ ]:
def convert_all_json_to_yolo():
    """
    10개 JSON → YOLO 형식 변환
    JSON에 명시된 split(train/val) 그대로 유지
    """
    setup_yolo_folders()

    # 통계 (split별)
    stats = {
        "train": {"success": 0, "no_face": 0, "no_image": 0, "invalid": 0},
        "val":   {"success": 0, "no_face": 0, "no_image": 0, "invalid": 0},
    }

    for (split, emotion), json_filename in JSON_FILES.items():
        json_path = os.path.join(JSON_DIR, json_filename)

        if not os.path.exists(json_path):
            print(f"⚠️  파일 없음: {json_path}")
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        images = data.get("images", [])

        for item in tqdm(images, desc=f"  {split}/{emotion}", leave=False):
            filename = item.get("filename")

            # 1. 얼굴 감지 안 됐으면 스킵
            annot = item.get("annot_HSEmotion", {})
            if not annot.get("face_detected", False):
                stats[split]["no_face"] += 1
                continue

            # 2. 이미지 파일 찾기
            img_path = find_image(filename, split, emotion)
            if img_path is None:
                stats[split]["no_image"] += 1
                continue

            # 3. 이미지 크기 확인
            try:
                with Image.open(img_path) as im:
                    img_w, img_h = im.size
            except Exception:
                stats[split]["invalid"] += 1
                continue

            # 4. bbox YOLO 변환
            bbox = item.get("bbox", {})
            if not bbox:
                stats[split]["invalid"] += 1
                continue

            try:
                cx, cy, w, h = convert_bbox_to_yolo(bbox, img_w, img_h)
            except Exception:
                stats[split]["invalid"] += 1
                continue

            if w <= 0 or h <= 0 or w > 1 or h > 1:
                stats[split]["invalid"] += 1
                continue

            # 5. 이미지 + 라벨 저장 (split은 JSON에 명시된 그대로)
            base_name = Path(filename).stem
            new_img_name = f"{emotion}_{base_name}.jpg"
            new_lbl_name = f"{emotion}_{base_name}.txt"

            dest_img = os.path.join(YOLO_DATASET_DIR, "images", split, new_img_name)
            dest_lbl = os.path.join(YOLO_DATASET_DIR, "labels", split, new_lbl_name)

            shutil.copy2(img_path, dest_img)
            with open(dest_lbl, "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

            stats[split]["success"] += 1

    # 결과 리포트
    print(f"""
╔═══════════════════════════════════════════════╗
║         JSON → YOLO 변환 완료                  ║
╠═══════════════════════════════════════════════╣
║  TRAIN                                         ║
║    성공:           {stats['train']['success']:>5}장                   ║
║    얼굴 미감지:    {stats['train']['no_face']:>5}장 (스킵)            ║
║    이미지 없음:    {stats['train']['no_image']:>5}장 (스킵)            ║
║    bbox 오류:      {stats['train']['invalid']:>5}장 (스킵)            ║
╠═══════════════════════════════════════════════╣
║  VAL                                           ║
║    성공:           {stats['val']['success']:>5}장                   ║
║    얼굴 미감지:    {stats['val']['no_face']:>5}장 (스킵)            ║
║    이미지 없음:    {stats['val']['no_image']:>5}장 (스킵)            ║
║    bbox 오류:      {stats['val']['invalid']:>5}장 (스킵)            ║
╠═══════════════════════════════════════════════╣
║  Train+Val 비율: {stats['train']['success']:>4} : {stats['val']['success']:>4}                 ║
╚═══════════════════════════════════════════════╝""")
    return stats

# 실행
stats = convert_all_json_to_yolo()

In [ ]:
# 변환 결과 검증
def verify_yolo_dataset():
    print("=" * 50)
    print("  YOLO 데이터셋 검증")
    print("=" * 50)

    for split in ["train", "val"]:
        img_dir = os.path.join(YOLO_DATASET_DIR, "images", split)
        lbl_dir = os.path.join(YOLO_DATASET_DIR, "labels", split)

        imgs = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        lbls = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]

        print(f"\n[{split}]")
        print(f"  이미지: {len(imgs)}장")
        print(f"  라벨:   {len(lbls)}개")

        # 감정별 분포
        from collections import Counter
        emotion_count = Counter(f.split("_")[0] for f in imgs)
        print(f"  감정별: {dict(emotion_count)}")

    # 샘플 라벨
    print(f"\n샘플 라벨 (train):")
    sample_dir = os.path.join(YOLO_DATASET_DIR, "labels", "train")
    samples = sorted(os.listdir(sample_dir))[:3]
    for fname in samples:
        with open(os.path.join(sample_dir, fname)) as f:
            content = f.read().strip()
        print(f"  {fname}: {content}")

verify_yolo_dataset()

## 3️⃣ YOLO 학습 설정 파일 생성

In [ ]:
yaml_content = f"""# YOLO Face Detection Dataset
path: {YOLO_DATASET_DIR}
train: images/train
val: images/val

# 클래스 정의 (얼굴 1개)
nc: 1
names: ['face']
"""

yaml_path = os.path.join(YOLO_DATASET_DIR, "face_data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"✅ YAML 생성: {yaml_path}\n")
print(yaml_content)

## 4️⃣ YOLO26 파인튜닝 학습

`yolo26n.pt` 베이스 → 우리 데이터로 파인튜닝

⏱️ 예상 시간: A40 기준 30분~1시간 (50 epochs)

In [ ]:
from ultralytics import YOLO

print("YOLO26n 베이스 모델 로드 중...")
model = YOLO("yolo26n.pt")

total_params = sum(p.numel() for p in model.model.parameters()) / 1e6
print(f"✅ 모델 로드 완료 (파라미터: ~{total_params:.1f}M)")

In [ ]:
# ═══════════════════════════════════════════════════
# 학습 시작
# ═══════════════════════════════════════════════════

results = model.train(
    data       = yaml_path,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    device     = DEVICE,
    project    = MODEL_OUTPUT_DIR,
    name       = "face_yolo26n",
    exist_ok   = True,

    # 데이터 증강 (극장 환경 대응)
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,        # 명도 (저조도 대응)
    degrees    = 10,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    mosaic     = 1.0,

    # 최적화
    optimizer  = "AdamW",
    lr0        = 0.001,
    weight_decay = 0.0005,
    patience   = 20,         # Early stopping
    save_period= 10,

    plots      = True,
    verbose    = True,
)

print("\n✅ 학습 완료!")

In [ ]:
import pandas as pd

result_dir       = os.path.join(MODEL_OUTPUT_DIR, "face_yolo26n")
best_model_path  = os.path.join(result_dir, "weights", "best.pt")

print(f"📁 결과 위치: {result_dir}")
print(f"🏆 Best 모델: {best_model_path}")

# 학습 메트릭
results_csv = os.path.join(result_dir, "results.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    print(f"\n📊 최종 epoch 메트릭:")
    last = df.iloc[-1]
    for col in ["metrics/mAP50(B)", "metrics/mAP50-95(B)",
                "metrics/precision(B)", "metrics/recall(B)"]:
        if col in df.columns:
            print(f"  {col}: {last[col]:.4f}")

    # Best epoch 찾기
    if "metrics/mAP50(B)" in df.columns:
        best_idx = df["metrics/mAP50(B)"].idxmax()
        print(f"\n🏆 Best epoch: {best_idx + 1}")
        print(f"   Best mAP50: {df.loc[best_idx, 'metrics/mAP50(B)']:.4f}")

## 5️⃣ 학습된 모델 테스트

In [ ]:
import cv2
import matplotlib.pyplot as plt

best_model = YOLO(best_model_path)

# val 폴더에서 무작위 6장 테스트
val_img_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")
test_imgs = sorted(os.listdir(val_img_dir))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, fname in enumerate(test_imgs):
    img_path = os.path.join(val_img_dir, fname)
    results = best_model(img_path, verbose=False)

    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(img, f"face {conf:.2f}", (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    axes[idx].imshow(img)
    axes[idx].set_title(fname[:25], fontsize=8)
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(result_dir, "test_results.png"), dpi=120)
plt.show()
print(f"\n✅ 테스트 결과 저장: {result_dir}/test_results.png")

## 6️⃣ val 데이터 평가

학습 안 한 val 데이터로 모델 정확도 확인

In [ ]:
# val 데이터로 평가
print("=" * 50)
print("  val 데이터 평가")
print("=" * 50)

metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE)

print(f"""
📊 val 데이터 평가 결과
───────────────────────────────
  mAP50:        {metrics.box.map50:.4f}
  mAP50-95:     {metrics.box.map:.4f}
  Precision:    {metrics.box.mp:.4f}
  Recall:       {metrics.box.mr:.4f}
""")

## 7️⃣ 메인 시스템에 적용

`emotion_engagement_system.ipynb`에서 모델 경로만 변경:

```python
# 기존
self.yolo = YOLO("yolov8n-face.pt")

# 변경
self.yolo = YOLO("/workspace/user5/yolo_finetuned/face_yolo26n/weights/best.pt")
```

In [ ]:
print("=" * 55)
print("  파인튜닝 완료 — 최종 모델 경로")
print("=" * 55)
print(f"\n{best_model_path}\n")
print("─" * 55)
print("메인 시스템 노트북에 붙여넣을 코드:")
print("─" * 55)
print(f'  self.yolo = YOLO("{best_model_path}")')